# GTFS2GMNS Verification Checklist

**Course:** CEE 412/598 Transportation Systems  
**Lecture:** 2.4 GTFS2GMNS User Guide  
**Purpose:** Step-by-step guide to verify GTFS2GMNS conversion

**⚠️ This notebook is designed for Google Colab**

This notebook will help you:
1. ✅ Clone the GTFS2GMNS repository and get Phoenix GTFS data
2. ✅ Check your GTFS input files
3. ✅ Run GTFS2GMNS conversion
4. ✅ Verify output files (node.csv and link.csv)
5. ✅ Answer all checklist questions

---

## Step 1: Setup Google Colab Environment

First, let's clone the repository and install necessary dependencies.

In [ ]:
# Install required packages (if not already installed)
!pip install pandas numpy -q

# Clone the repository (skip if already exists)
import os
if not os.path.exists('gtfs2gmns'):
    !git clone https://github.com/itsfangtang/gtfs2gmns
else:
    print("📁 Repository already exists, skipping clone...")

# Change to the project directory
%cd gtfs2gmns

print("✅ Repository ready!")
print(f"Current directory: {os.getcwd()}")

## Step 2: Import Libraries and Configure Parameters

Now let's import the necessary libraries and configure the GTFS2GMNS parameters.

**⚠️ IMPORTANT:** The paths are already configured for Phoenix dataset. You can modify these if needed:

1. **`input_path`**: Path to your GTFS data folder (default: Phoenix)
2. **`output_path`**: Path where GMNS output files will be saved
3. **`time_period`**: Time window in 'HHMM_HHMM' format (e.g., '0700_0800' for 7:00-8:00 AM)

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

# ============================================
# CONFIGURATION: Paths for Phoenix dataset
# ============================================

# Phoenix dataset (single agency)
input_path = './GTFS/Phoenix'          # GTFS data folder
output_path = './GMNS/Phoenix'          # Output folder for GMNS files
time_period = '0700_0800'               # Time window: HHMM_HHMM format (7:00-8:00 AM)

# Alternative: Multiple Agencies (San Francisco)
# input_path = './GTFS/SF'             # Contains multiple agency subdirectories
# output_path = './GMNS/SF'
# time_period = '1700_1800'            # Evening peak: 5:00 PM - 6:00 PM

# Verify paths exist
base_dir = os.getcwd()
input_full_path = os.path.join(base_dir, input_path.lstrip('./'))
output_full_path = os.path.join(base_dir, output_path.lstrip('./'))

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Base directory: {base_dir}")
print(f"Input path: {input_path}")
print(f"Input full path: {input_full_path}")
print(f"Output path: {output_path}")
print(f"Output full path: {output_full_path}")
print(f"Time period: {time_period}")
print("=" * 60)

# Create output directory if it doesn't exist
os.makedirs(output_full_path, exist_ok=True)
print(f"\n✅ Output directory created/verified: {output_full_path}")

## Step 3: Check GTFS Input Files

Before running GTFS2GMNS, we need to verify that all required GTFS files exist and are properly formatted.

**Required GTFS files:**
- `agency.txt`
- `stops.txt`
- `routes.txt`
- `trips.txt`
- `stop_times.txt`

In [ ]:
def check_gtfs_files(gtfs_path):
    """Check if required GTFS files exist and display basic information."""
    
    required_files = ['agency.txt', 'stops.txt', 'routes.txt', 'trips.txt', 'stop_times.txt']
    optional_files = ['calendar.txt', 'calendar_dates.txt', 'shapes.txt', 'frequencies.txt']
    
    print("=" * 60)
    print("GTFS INPUT FILE VERIFICATION")
    print("=" * 60)
    print(f"Checking directory: {gtfs_path}\n")
    
    # Check if directory exists
    if not os.path.exists(gtfs_path):
        print(f"❌ ERROR: Directory '{gtfs_path}' does not exist!")
        print("Please check your input_path configuration.")
        return False
    
    # Check required files
    print("REQUIRED FILES:")
    all_required_exist = True
    for file in required_files:
        file_path = os.path.join(gtfs_path, file)
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path) / 1024  # Size in KB
            print(f"  ✅ {file:20s} ({file_size:.1f} KB)")
            
            # Try to read and show basic stats
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig')
                print(f"      → {len(df)} records, {len(df.columns)} columns")
            except Exception as e:
                print(f"      ⚠️  Warning: Could not read file ({str(e)[:50]})")
        else:
            print(f"  ❌ {file:20s} MISSING!")
            all_required_exist = False
    
    print("\nOPTIONAL FILES:")
    for file in optional_files:
        file_path = os.path.join(gtfs_path, file)
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path) / 1024
            print(f"  ✓ {file:20s} ({file_size:.1f} KB)")
        else:
            print(f"  - {file:20s} (not present)")
    
    print("\n" + "=" * 60)
    if all_required_exist:
        print("✅ All required GTFS files are present!")
        return True
    else:
        print("❌ Some required files are missing. Please check your GTFS data.")
        return False

# Run the check
gtfs_valid = check_gtfs_files(input_path)

### Step 3.1: Examine GTFS File Contents

Let's look at the structure and key fields of each GTFS file to understand the data.

In [ ]:
def examine_gtfs_file(gtfs_path, filename, key_fields=None):
    """Examine a GTFS file and display key information."""
    file_path = os.path.join(gtfs_path, filename)
    
    if not os.path.exists(file_path):
        print(f"❌ {filename} not found!")
        return None
    
    try:
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        
        print(f"\n{'='*60}")
        print(f"{filename}")
        print(f"{'='*60}")
        print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"\nColumns: {list(df.columns)}")
        
        if key_fields:
            print(f"\nKey Fields Check:")
            for field in key_fields:
                if field in df.columns:
                    print(f"  ✅ {field}")
                else:
                    print(f"  ❌ {field} - MISSING!")
        
        print(f"\nFirst 3 rows:")
        display(df.head(3))
        
        return df
    except Exception as e:
        print(f"❌ Error reading {filename}: {str(e)}")
        return None

# Examine each required file
print("Examining GTFS files...")
agency_df = examine_gtfs_file(input_path, 'agency.txt', key_fields=['agency_name'])
stops_df = examine_gtfs_file(input_path, 'stops.txt', key_fields=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
routes_df = examine_gtfs_file(input_path, 'routes.txt', key_fields=['route_id', 'route_short_name', 'route_long_name', 'route_type'])
trips_df = examine_gtfs_file(input_path, 'trips.txt', key_fields=['trip_id', 'route_id'])
stop_times_df = examine_gtfs_file(input_path, 'stop_times.txt', key_fields=['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'stop_sequence'])

In [ ]:
def calculate_input_statistics(gtfs_path):
    """Calculate key statistics from GTFS files."""
    
    stats = {}
    
    # Agency name
    try:
        agency_df = pd.read_csv(os.path.join(gtfs_path, 'agency.txt'), encoding='utf-8-sig')
        stats['agency_name'] = agency_df['agency_name'].iloc[0] if 'agency_name' in agency_df.columns else 'N/A'
    except:
        stats['agency_name'] = 'N/A'
    
    # Number of stops
    try:
        stops_df = pd.read_csv(os.path.join(gtfs_path, 'stops.txt'), encoding='utf-8-sig')
        stats['num_stops'] = len(stops_df)
    except:
        stats['num_stops'] = 0
    
    # Number of routes
    try:
        routes_df = pd.read_csv(os.path.join(gtfs_path, 'routes.txt'), encoding='utf-8-sig')
        stats['num_routes'] = len(routes_df)
    except:
        stats['num_routes'] = 0
    
    # Number of trips
    try:
        trips_df = pd.read_csv(os.path.join(gtfs_path, 'trips.txt'), encoding='utf-8-sig')
        stats['num_trips'] = len(trips_df)
    except:
        stats['num_trips'] = 0
    
    # Number of stop_time records
    try:
        stop_times_df = pd.read_csv(os.path.join(gtfs_path, 'stop_times.txt'), encoding='utf-8-sig')
        stats['num_stop_times'] = len(stop_times_df)
    except:
        stats['num_stop_times'] = 0
    
    return stats

# Calculate and display statistics
input_stats = calculate_input_statistics(input_path)

print("=" * 60)
print("GTFS INPUT STATISTICS")
print("=" * 60)
for key, value in input_stats.items():
    print(f"{key:20s}: {value}")
print("=" * 60)
print("\n💡 Save these statistics! You'll need them to verify the output.")

In [ ]:
# Import the gtfs2gmns module
import sys
sys.path.append('.')

# Import necessary functions and modules
import gtfs2gmns
from gtfs2gmns import gtfs2gmns
import datetime

# Define the time conversion function (same as in gtfs2gmns.py)
def _hhmm_to_minutes(time_period):
    from_time = datetime.time(int(time_period[0:2]), int(time_period[2:4]))
    to_time = datetime.time(int(time_period[-4:-2]), int(time_period[-2:]))
    from_time_min = from_time.hour * 60 + from_time.minute
    to_time_min = to_time.hour * 60 + to_time.minute
    return from_time_min, to_time_min

# Set global variables for time period (required by reading_data function)
gtfs2gmns.period_start_time, gtfs2gmns.period_end_time = _hhmm_to_minutes(time_period)

# Run the conversion
print("=" * 60)
print("RUNNING GTFS2GMNS CONVERSION")
print("=" * 60)
print(f"Input: {input_path}")
print(f"Output: {output_path}")
print(f"Time period: {time_period}")
print(f"Time window: {gtfs2gmns.period_start_time} - {gtfs2gmns.period_end_time} minutes")
print("=" * 60)
print("\n⏳ Starting conversion... This may take a few minutes.\n")

try:
    gtfs2gmns(input_path, output_path)
    print("\n" + "=" * 60)
    print("✅ Conversion completed successfully!")
    print("=" * 60)
    print(f"\n📁 Output files should be in: {output_path}")
    print("   - node.csv")
    print("   - link.csv")
except Exception as e:
    print(f"\n❌ Error during conversion: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check that GTFS files exist in the input path")
    print("2. Verify the time period format is correct (HHMM_HHMM)")
    print("3. Check the error message above for details")
    import traceback
    traceback.print_exc()

## Step 4: Run GTFS2GMNS Conversion

In Google Colab, we can run the conversion directly from the notebook. The script will use the parameters we configured in Step 2.

**Note:** The conversion may take a few minutes depending on the size of your GTFS data. You should see progress messages printed to the console.

## Step 5: Verify GMNS Output Files

**Checklist Question Q11:** GTFS2GMNS produces two output files: `node.csv` and `link.csv`

Let's verify these files exist and examine their structure.

In [ ]:
def check_output_files(output_path):
    """Check if GMNS output files exist."""
    
    node_file = os.path.join(output_path, 'node.csv')
    link_file = os.path.join(output_path, 'link.csv')
    
    print("=" * 60)
    print("GMNS OUTPUT FILE VERIFICATION")
    print("=" * 60)
    print(f"Checking directory: {output_path}\n")
    
    if not os.path.exists(output_path):
        print(f"❌ ERROR: Output directory '{output_path}' does not exist!")
        print("Please run GTFS2GMNS conversion first.")
        return False, False
    
    node_exists = os.path.exists(node_file)
    link_exists = os.path.exists(link_file)
    
    if node_exists:
        node_size = os.path.getsize(node_file) / 1024  # KB
        print(f"✅ node.csv exists ({node_size:.1f} KB)")
    else:
        print(f"❌ node.csv MISSING!")
    
    if link_exists:
        link_size = os.path.getsize(link_file) / 1024  # KB
        print(f"✅ link.csv exists ({link_size:.1f} KB)")
    else:
        print(f"❌ link.csv MISSING!")
    
    print("\n" + "=" * 60)
    
    if node_exists and link_exists:
        print("✅ Both output files are present!")
        print("✅ Q11 ANSWER: GTFS2GMNS produces two output files: node.csv and link.csv")
        return True, True
    else:
        print("❌ Some output files are missing. Please check the conversion.")
        return node_exists, link_exists

# Check output files
node_ok, link_ok = check_output_files(output_path)

## Step 6: Examine node.csv Structure

**Checklist Question Q12:** List the four types of nodes (actually 2 main types: Physical and Service nodes)

In [ ]:
if node_ok:
    node_file = os.path.join(output_path, 'node.csv')
    node_df = pd.read_csv(node_file)
    
    print("=" * 60)
    print("NODE.CSV STRUCTURE")
    print("=" * 60)
    print(f"Shape: {node_df.shape[0]} rows × {node_df.shape[1]} columns")
    print(f"\nColumns ({len(node_df.columns)}):")
    for i, col in enumerate(node_df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    print("\n" + "=" * 60)
    print("NODE TYPES")
    print("=" * 60)
    
    # Identify node types
    if 'node_type' in node_df.columns:
        node_types = node_df['node_type'].value_counts()
        print("\nNode Type Distribution:")
        for node_type, count in node_types.items():
            print(f"  • {node_type:30s}: {count:5d} nodes")
        
        # Categorize into physical vs service nodes
        physical_nodes = node_df[node_df['node_id'] == node_df['physical_node_id']]
        service_nodes = node_df[node_df['node_id'] != node_df['physical_node_id']]
        
        print("\n" + "-" * 60)
        print("ANSWER TO Q12: Node Types")
        print("-" * 60)
        print("1. Physical nodes (stops/stations):")
        print(f"   → {len(physical_nodes)} physical nodes")
        if 'node_type' in physical_nodes.columns:
            physical_types = physical_nodes['node_type'].unique()
            for ptype in physical_types:
                count = len(physical_nodes[physical_nodes['node_type'] == ptype])
                print(f"     - {ptype}: {count} nodes")
        
        print("\n2. Service nodes (route-specific service points):")
        print(f"   → {len(service_nodes)} service nodes")
        if 'node_type' in service_nodes.columns:
            service_types = service_nodes['node_type'].unique()
            for stype in service_types:
                count = len(service_nodes[service_nodes['node_type'] == stype])
                print(f"     - {stype}: {count} nodes")
    
    print("\n" + "=" * 60)
    print("SAMPLE NODES")
    print("=" * 60)
    
    # Show sample physical nodes
    print("\nSample Physical Nodes:")
    display(physical_nodes[['node_id', 'name', 'node_type', 'x_coord', 'y_coord', 'route_type']].head(5))
    
    # Show sample service nodes
    print("\nSample Service Nodes:")
    display(service_nodes[['node_id', 'physical_node_id', 'name', 'node_type', 'directed_route_id']].head(5))
    
    print("\n✅ Q12 ANSWER: See node types above")
else:
    print("❌ node.csv not found. Please run GTFS2GMNS conversion first.")

## Step 7: Examine link.csv Structure

**Checklist Question Q13:** List the four types of links  
**Checklist Question Q15:** What field in link.csv contains travel time?  
**Checklist Question Q16:** What field in link.csv contains vehicle/link capacity?

In [ ]:
if link_ok:
    link_file = os.path.join(output_path, 'link.csv')
    link_df = pd.read_csv(link_file)
    
    print("=" * 60)
    print("LINK.CSV STRUCTURE")
    print("=" * 60)
    print(f"Shape: {link_df.shape[0]} rows × {link_df.shape[1]} columns")
    print(f"\nColumns ({len(link_df.columns)}):")
    for i, col in enumerate(link_df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    print("\n" + "=" * 60)
    print("LINK TYPES")
    print("=" * 60)
    
    # Identify link types
    if 'link_type_name' in link_df.columns:
        link_types = link_df['link_type_name'].value_counts()
        print("\nLink Type Distribution:")
        for link_type, count in link_types.items():
            print(f"  • {link_type:30s}: {count:5d} links")
        
        print("\n" + "-" * 60)
        print("ANSWER TO Q13: Four Types of Links")
        print("-" * 60)
        print("1. Service links: Connect consecutive stops along routes")
        service_count = len(link_df[link_df['link_type_name'] == 'service_links'])
        print(f"   → {service_count} service links")
        
        print("2. Boarding links: Connect physical stations to service nodes (passenger boards)")
        boarding_count = len(link_df[link_df['link_type_name'] == 'boarding_links'])
        print(f"   → {boarding_count} boarding links")
        
        print("3. Deboarding links: Connect service nodes to physical stations (passenger alights)")
        deboarding_count = len(link_df[link_df['link_type_name'] == 'deboarding_links']) if 'deboarding_links' in link_df['link_type_name'].values else 0
        print(f"   → {deboarding_count} deboarding links")
        
        print("4. Transferring links: Connect nearby physical stations (walk between stops)")
        transfer_count = len(link_df[link_df['link_type_name'] == 'transferring_links'])
        print(f"   → {transfer_count} transferring links")
    
    # Check for key fields
    print("\n" + "=" * 60)
    print("KEY FIELDS CHECK")
    print("=" * 60)
    
    key_fields = {
        'VDF_fftt1': 'Travel time (minutes)',
        'capacity': 'Vehicle/link capacity',
        'length': 'Link length (meters)',
        'free_speed': 'Free-flow speed (km/h)',
        'VDF_penalty1': 'Transfer penalty (minutes)'
    }
    
    for field, description in key_fields.items():
        if field in link_df.columns:
            print(f"✅ {field:15s}: {description}")
            # Show sample values
            sample_values = link_df[field].dropna().head(3).tolist()
            print(f"   Sample values: {sample_values}")
        else:
            print(f"❌ {field:15s}: MISSING!")
    
    print("\n" + "-" * 60)
    print("ANSWERS TO CHECKLIST QUESTIONS")
    print("-" * 60)
    print("✅ Q15 ANSWER: VDF_fftt1 contains travel time (in minutes)")
    print("✅ Q16 ANSWER: capacity contains vehicle/link capacity")
    
    print("\n" + "=" * 60)
    print("SAMPLE LINKS BY TYPE")
    print("=" * 60)
    
    # Show sample service links
    if 'service_links' in link_df['link_type_name'].values:
        print("\nSample Service Links:")
        service_links = link_df[link_df['link_type_name'] == 'service_links']
        display(service_links[['link_id', 'from_node_id', 'to_node_id', 'link_type_name', 'VDF_fftt1', 'capacity', 'directed_route_id']].head(3))
    
    # Show sample boarding links
    if 'boarding_links' in link_df['link_type_name'].values:
        print("\nSample Boarding Links:")
        boarding_links = link_df[link_df['link_type_name'] == 'boarding_links']
        display(boarding_links[['link_id', 'from_node_id', 'to_node_id', 'link_type_name', 'VDF_fftt1']].head(3))
    
    # Show sample transfer links
    if 'transferring_links' in link_df['link_type_name'].values:
        print("\nSample Transferring Links:")
        transfer_links = link_df[link_df['link_type_name'] == 'transferring_links']
        display(transfer_links[['link_id', 'from_node_id', 'to_node_id', 'link_type_name', 'length', 'VDF_penalty1']].head(3))
else:
    print("❌ link.csv not found. Please run GTFS2GMNS conversion first.")

## Step 8: Verify Output Statistics

Compare the output statistics with your input statistics to ensure the conversion worked correctly.

**Checklist Question Q14:** In your example (Phoenix), how many physical stations? Routes?

In [ ]:
def calculate_output_statistics(output_path):
    """Calculate key statistics from GMNS output files."""
    
    stats = {}
    
    node_file = os.path.join(output_path, 'node.csv')
    link_file = os.path.join(output_path, 'link.csv')
    
    if os.path.exists(node_file):
        node_df = pd.read_csv(node_file)
        
        # Physical nodes (where node_id == physical_node_id)
        physical_nodes = node_df[node_df['node_id'] == node_df['physical_node_id']]
        stats['num_physical_nodes'] = len(physical_nodes)
        
        # Service nodes (where node_id != physical_node_id)
        service_nodes = node_df[node_df['node_id'] != node_df['physical_node_id']]
        stats['num_service_nodes'] = len(service_nodes)
        
        # Total nodes
        stats['num_total_nodes'] = len(node_df)
        
        # Unique routes
        if 'directed_route_id' in node_df.columns:
            unique_routes = node_df['directed_route_id'].dropna().nunique()
            stats['num_directed_routes'] = unique_routes
        
        # Agency name
        if 'agency_name' in node_df.columns:
            stats['agency_name'] = node_df['agency_name'].iloc[0] if len(node_df) > 0 else 'N/A'
    
    if os.path.exists(link_file):
        link_df = pd.read_csv(link_file)
        
        # Total links
        stats['num_total_links'] = len(link_df)
        
        # Links by type
        if 'link_type_name' in link_df.columns:
            link_type_counts = link_df['link_type_name'].value_counts().to_dict()
            stats['link_type_counts'] = link_type_counts
        
        # Service links
        service_links = link_df[link_df['link_type_name'] == 'service_links']
        stats['num_service_links'] = len(service_links)
        
        # Boarding links
        boarding_links = link_df[link_df['link_type_name'] == 'boarding_links']
        stats['num_boarding_links'] = len(boarding_links)
        
        # Transfer links
        transfer_links = link_df[link_df['link_type_name'] == 'transferring_links']
        stats['num_transfer_links'] = len(transfer_links)
    
    return stats

# Calculate output statistics
output_stats = calculate_output_statistics(output_path)

print("=" * 60)
print("GMNS OUTPUT STATISTICS")
print("=" * 60)
for key, value in output_stats.items():
    if key != 'link_type_counts':
        print(f"{key:25s}: {value}")

if 'link_type_counts' in output_stats:
    print("\nLink Type Breakdown:")
    for link_type, count in output_stats['link_type_counts'].items():
        print(f"  {link_type:30s}: {count:5d}")

print("\n" + "=" * 60)
print("COMPARISON: INPUT vs OUTPUT")
print("=" * 60)

if 'input_stats' in locals() and 'output_stats' in locals():
    print(f"\nInput Statistics:")
    print(f"  Number of stops (GTFS):     {input_stats.get('num_stops', 'N/A')}")
    print(f"  Number of routes (GTFS):    {input_stats.get('num_routes', 'N/A')}")
    
    print(f"\nOutput Statistics:")
    print(f"  Physical nodes (GMNS):      {output_stats.get('num_physical_nodes', 'N/A')}")
    print(f"  Service nodes (GMNS):       {output_stats.get('num_service_nodes', 'N/A')}")
    print(f"  Directed routes (GMNS):      {output_stats.get('num_directed_routes', 'N/A')}")
    
    # Verification
    print(f"\n✅ Verification:")
    if input_stats.get('num_stops', 0) > 0:
        if abs(input_stats['num_stops'] - output_stats.get('num_physical_nodes', 0)) <= 5:  # Allow small difference
            print(f"  ✅ Physical nodes match GTFS stops (within tolerance)")
        else:
            print(f"  ⚠️  Physical nodes ({output_stats.get('num_physical_nodes', 0)}) differ from GTFS stops ({input_stats.get('num_stops', 0)})")
            print(f"     (This may be normal if some stops are filtered)")

print("\n✅ Q14 ANSWER: Check your output statistics above")

## Step 9: Answer Linear Programming Question (Q17)

**Checklist Question Q17:** From Section 6.5, the transit assignment LP minimizes total __________ subject to flow conservation and __________ constraints.

In [ ]:
print("=" * 60)
print("ANSWER TO Q17: Linear Programming Formulation")
print("=" * 60)
print("")
print("From Section 6.5 (Linear Programming Applications) of the User Guide:")
print("")
print("The transit assignment LP:")
print("  • Minimizes total TRAVEL TIME")
print("  • Subject to:")
print("    1. Flow conservation constraints (mass balance at each node)")
print("    2. CAPACITY constraints (link flows cannot exceed capacity)")
print("")
print("Key fields from link.csv used in LP:")
print("  • VDF_fftt1: Travel time (used in objective function)")
print("  • capacity: Maximum flow (used in capacity constraints)")
print("  • from_node_id, to_node_id: Network structure (used in flow conservation)")
print("")
print("✅ Q17 ANSWER:")
print("   The transit assignment LP minimizes total TRAVEL TIME")
print("   subject to flow conservation and CAPACITY constraints.")

## Step 10: Summary Checklist Answers

Let's compile all the answers to the checklist questions from Lecture 2.4.

In [ ]:
print("=" * 60)
print("CHECKLIST ANSWERS SUMMARY (Lecture 2.4)")
print("=" * 60)
print("")
print("Q11. GTFS2GMNS produces two output files:")
print("     ANSWER: node.csv and link.csv")
print("")
print("Q12. List the four types of nodes:")
print("     ANSWER:")
print("     1. Physical nodes (stops/stations)")
print("     2. Service nodes (route-specific service points)")
print("        - bus_service_node")
print("        - metro_service_node")
print("        - rail_service_node")
print("        - tram_service_node")
print("")
print("Q13. List the four types of links:")
print("     ANSWER:")
print("     1. Service links (connect stops along routes)")
print("     2. Boarding links (passenger boards)")
print("     3. Deboarding links (passenger alights)")
print("     4. Transferring links (walk between stops)")
print("")
print("Q14. In your example (Phoenix), how many physical stations? Routes?")
if 'output_stats' in locals():
    print(f"     ANSWER: {output_stats.get('num_physical_nodes', 'N/A')} physical stations, {output_stats.get('num_directed_routes', 'N/A')} directed routes")
else:
    print("     ANSWER: Check your output statistics above")
print("     (Your actual numbers will be displayed in Step 8)")
print("")
print("Q15. What field in link.csv contains travel time?")
print("     ANSWER: VDF_fftt1")
print("")
print("Q16. What field in link.csv contains vehicle/link capacity?")
print("     ANSWER: capacity")
print("")
print("Q17. From Section 6.5, the transit assignment LP minimizes total")
print("     __________ subject to flow conservation and __________ constraints.")
print("     ANSWER: TRAVEL TIME, CAPACITY")
print("")
print("=" * 60)
print("✅ All checklist questions answered!")
print("=" * 60)

In [ ]:
# Option 1: Download individual files
from google.colab import files

# Download node.csv
if os.path.exists(os.path.join(output_path, 'node.csv')):
    files.download(os.path.join(output_path, 'node.csv'))
    print("✅ node.csv downloaded")

# Download link.csv
if os.path.exists(os.path.join(output_path, 'link.csv')):
    files.download(os.path.join(output_path, 'link.csv'))
    print("✅ link.csv downloaded")

# Option 2: Create a zip file and download
# Uncomment the following lines if you want to download as a zip file:
# import shutil
# zip_path = 'GMNS_Phoenix_output.zip'
# shutil.make_archive('GMNS_Phoenix_output', 'zip', output_path)
# files.download(zip_path)
# print("✅ Output files zipped and downloaded")

---

## ✅ Completion Checklist

Before submitting, make sure you have:

- [ ] ✅ Cloned the repository and verified GTFS files exist
- [ ] ✅ Run GTFS2GMNS conversion successfully
- [ ] ✅ Verified node.csv and link.csv are generated
- [ ] ✅ Examined node types (physical and service nodes)
- [ ] ✅ Examined link types (service, boarding, deboarding, transferring)
- [ ] ✅ Verified output statistics match input statistics
- [ ] ✅ Answered all checklist questions (Q11-Q17)
- [ ] ✅ Completed validation checklist

**Congratulations!** You have successfully completed the GTFS2GMNS verification process. 🎉

---

## 📝 Notes for Google Colab Users

- **Session Timeout:** Colab sessions may timeout after inactivity. If this happens, re-run all cells from the beginning.
- **File Persistence:** Files in Colab are temporary. If you need to keep the output files, download them using Step 12.
- **Restart Runtime:** If you encounter any issues, try restarting the runtime (Runtime → Restart runtime) and re-run all cells.

## Step 11: Final Validation Checklist

Use this final checklist to ensure your conversion is correct.

In [ ]:
def validation_checklist(output_path, input_stats=None):
    """Run final validation checklist."""
    
    print("=" * 60)
    print("FINAL VALIDATION CHECKLIST")
    print("=" * 60)
    print("")
    
    checks = []
    
    # Check 1: Output files exist
    node_file = os.path.join(output_path, 'node.csv')
    link_file = os.path.join(output_path, 'link.csv')
    
    if os.path.exists(node_file) and os.path.exists(link_file):
        print("✅ [1/8] Output files exist (node.csv and link.csv)")
        checks.append(True)
    else:
        print("❌ [1/8] Output files missing")
        checks.append(False)
    
    # Check 2: node.csv structure
    if os.path.exists(node_file):
        node_df = pd.read_csv(node_file)
        required_node_fields = ['node_id', 'physical_node_id', 'name', 'x_coord', 'y_coord', 'node_type']
        missing = [f for f in required_node_fields if f not in node_df.columns]
        if len(missing) == 0:
            print("✅ [2/8] node.csv has all required fields")
            checks.append(True)
        else:
            print(f"❌ [2/8] node.csv missing fields: {missing}")
            checks.append(False)
    
    # Check 3: link.csv structure
    if os.path.exists(link_file):
        link_df = pd.read_csv(link_file)
        required_link_fields = ['link_id', 'from_node_id', 'to_node_id', 'link_type', 'VDF_fftt1', 'capacity']
        missing = [f for f in required_link_fields if f not in link_df.columns]
        if len(missing) == 0:
            print("✅ [3/8] link.csv has all required fields")
            checks.append(True)
        else:
            print(f"❌ [3/8] link.csv missing fields: {missing}")
            checks.append(False)
    
    # Check 4: Physical nodes exist
    if os.path.exists(node_file):
        node_df = pd.read_csv(node_file)
        physical_nodes = node_df[node_df['node_id'] == node_df['physical_node_id']]
        if len(physical_nodes) > 0:
            print(f"✅ [4/8] Physical nodes present ({len(physical_nodes)} nodes)")
            checks.append(True)
        else:
            print("❌ [4/8] No physical nodes found")
            checks.append(False)
    
    # Check 5: Service nodes exist
    if os.path.exists(node_file):
        node_df = pd.read_csv(node_file)
        service_nodes = node_df[node_df['node_id'] != node_df['physical_node_id']]
        if len(service_nodes) > 0:
            print(f"✅ [5/8] Service nodes present ({len(service_nodes)} nodes)")
            checks.append(True)
        else:
            print("❌ [5/8] No service nodes found")
            checks.append(False)
    
    # Check 6: All link types present
    if os.path.exists(link_file):
        link_df = pd.read_csv(link_file)
        if 'link_type_name' in link_df.columns:
            link_types = link_df['link_type_name'].unique()
            required_types = ['service_links', 'boarding_links', 'transferring_links']
            found_types = [t for t in required_types if t in link_types]
            if len(found_types) >= 2:  # At least service and boarding
                print(f"✅ [6/8] Link types present: {found_types}")
                checks.append(True)
            else:
                print(f"⚠️  [6/8] Some link types missing. Found: {link_types}")
                checks.append(True)  # Not critical
        else:
            print("❌ [6/8] link_type_name field missing")
            checks.append(False)
    
    # Check 7: Data consistency
    if os.path.exists(node_file) and os.path.exists(link_file):
        node_df = pd.read_csv(node_file)
        link_df = pd.read_csv(link_file)
        
        # Check that link node IDs exist in node.csv
        all_node_ids = set(node_df['node_id'].unique())
        link_from_nodes = set(link_df['from_node_id'].unique())
        link_to_nodes = set(link_df['to_node_id'].unique())
        
        missing_from = link_from_nodes - all_node_ids
        missing_to = link_to_nodes - all_node_ids
        
        if len(missing_from) == 0 and len(missing_to) == 0:
            print("✅ [7/8] Link node IDs match node.csv (data consistency)")
            checks.append(True)
        else:
            print(f"⚠️  [7/8] Some link node IDs not found in node.csv")
            print(f"     Missing from_nodes: {len(missing_from)}, Missing to_nodes: {len(missing_to)}")
            checks.append(False)
    
    # Check 8: Statistics match (if input stats available)
    if input_stats and 'num_stops' in input_stats:
        if os.path.exists(node_file):
            node_df = pd.read_csv(node_file)
            physical_nodes = node_df[node_df['node_id'] == node_df['physical_node_id']]
            num_physical = len(physical_nodes)
            num_gtfs_stops = input_stats['num_stops']
            
            if abs(num_physical - num_gtfs_stops) <= 10:  # Allow some tolerance
                print(f"✅ [8/8] Physical nodes match GTFS stops (within tolerance)")
                print(f"     GTFS stops: {num_gtfs_stops}, Physical nodes: {num_physical}")
                checks.append(True)
            else:
                print(f"⚠️  [8/8] Physical nodes differ from GTFS stops")
                print(f"     GTFS stops: {num_gtfs_stops}, Physical nodes: {num_physical}")
                checks.append(True)  # Not necessarily an error
        else:
            checks.append(False)
    else:
        print("⚠️  [8/8] Cannot verify statistics (input stats not available)")
        checks.append(True)
    
    print("")
    print("=" * 60)
    passed = sum(checks)
    total = len(checks)
    print(f"VALIDATION RESULT: {passed}/{total} checks passed")
    
    if passed == total:
        print("✅ All validation checks passed! Your conversion is successful.")
    elif passed >= total * 0.75:
        print("⚠️  Most checks passed. Review any warnings above.")
    else:
        print("❌ Multiple checks failed. Please review your conversion.")
    
    print("=" * 60)
    
    return checks

# Run validation
validation_results = validation_checklist(output_path, input_stats if 'input_stats' in locals() else None)

---

## ✅ Completion Checklist

Before submitting, make sure you have:

- [ ] ✅ Verified all GTFS input files exist
- [ ] ✅ Run GTFS2GMNS conversion successfully
- [ ] ✅ Verified node.csv and link.csv are generated
- [ ] ✅ Examined node types (physical and service nodes)
- [ ] ✅ Examined link types (service, boarding, deboarding, transferring)
- [ ] ✅ Verified output statistics match input statistics
- [ ] ✅ Answered all checklist questions (Q11-Q17)
- [ ] ✅ Completed validation checklist

**Congratulations!** You have successfully completed the GTFS2GMNS verification process. 🎉